In [1]:
import signac
import sys
import os
import numpy as np
import pandas as pd

from utils.molec_class_files import esolvs
from Build_GPs.utils.id_new_samples import new_samples_vle, find_pareto, new_samples_ld, check_mse_10
from Build_GPs.utils.models import get_best_models
from Build_GPs.utils.plot import plot_gp_examples

os.chdir("/scratch365/mcarlozo/ES-FFO/Build_GPs")

#Set iters to analyze and properties to analyze
mol_names = ["R125"] # Change me as needed
molec_dict = esolvs.make_dict(mol_names)
#Set seeds and preferences
cl_shuffle_seed = 1  # classifier
gp_shuffle_seed = 42  # GP seed
dist_seed = 1  # Distance seed
mse_less_10_thresh = 25
save_csv = True
save_fig = True
verbose = True
iter = 4

2025-05-22 15:02:52.191549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747940572.211469  841869 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747940572.217670  841869 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747940572.234409  841869 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747940572.234438  841869 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747940572.234440  841869 computation_placer.cc:177] computation placer alr

In [2]:
# iter_type = "vle_iters"
# property_names = ["liq_density", "vap_density", "Pvap", "Hvap"]
iter_type = "ld_iters"
property_names = ["liq_density"]

In [ ]:
df_all = pd.DataFrame()
for i in range(1,iter+1):
    file = f"analysis/R125/{iter_type}/iter-{i}/results.csv"
    df = pd.read_csv(file, index_col=0, header=0)
    df["iter"] = i
    if i == 1:
        df_all = df
    else:
        df_all = pd.concat([df_all, df], ignore_index=True)

if "density" in df_all.columns:
    df_all.rename(columns={"density": "liq_density"}, inplace=True)
if "Hvap" in property_names:
    df_all["Hvap"] = df_all["Hvap"] * 1000/molec_dict["R125"].molecular_weight  # Convert to J/mol
df_all_molec = {"R125":df_all}

   sigma_C1  sigma_C2  sigma_F1  sigma_F2  sigma_H1  epsilon_C1  epsilon_C2  \
0   0.30048   0.38676    0.2959   0.31283    0.1805    0.312358    0.331414   
1   0.30048   0.38676    0.2959   0.31283    0.1805    0.312358    0.331414   
2   0.30048   0.38676    0.2959   0.31283    0.1805    0.312358    0.331414   
3   0.30048   0.38676    0.2959   0.31283    0.1805    0.312358    0.331414   
4   0.30048   0.38676    0.2959   0.31283    0.1805    0.312358    0.331414   

   epsilon_F1  epsilon_F2  epsilon_H1  temperature  liq_density  iter  
0    0.148891    0.236505    0.036457          229  1342.450178     1  
1    0.148891    0.236505    0.036457          249  1243.019150     1  
2    0.148891    0.236505    0.036457          269  1136.867265     1  
3    0.148891    0.236505    0.036457          289    56.355480     1  
4    0.148891    0.236505    0.036457          309   104.164059     1  


In [ ]:
models_molecs = get_best_models(df_all_molec, molec_dict, iter_type, gp_shuffle_seed)
# plot_gp_examples(df_all_molec, molec_dict, iter_type, gp_shuffle_seed, save_fig)

2025-05-22 15:03:05.200842: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
#Check VLE
#Check pareto efficient samples for each molecule to see if there is one with < 5% error in all properties
# all_final_params = find_pareto(df_all_molec, molec_dict, property_names)
# for key, value in all_final_params.items():
#     #If there are, we have the final parameters
#     if len(value) > 0:
#         print(f"{key}: Final parameters:")
#         print(value)
#     #Otherwise we need to move to the next iteration
#     else:
#         print(f"{key} : No final parameters found. Move to iteration {max(iters) + 1}")
#         next_samples = new_samples_vle(df_all_molec, molec_dict, verbose, False, gp_shuffle_seed, dist_seed)

#Check LD
#Check the MSE of the new samples
mse_less10 = check_mse_10(df_all_molec, molec_dict, mse_less_10_thresh, dist_seed, save_csv)
#Find the next samples to run if fewer than 25 samples have MSE less than 10
print(mse_less10["R125"])
for key, value in mse_less10.items():
    if len(value) >= mse_less_10_thresh:
        print(f"{key} : All samples have MSE less than {mse_less_10_thresh}. Proceed to VLE.")
    else:
        print(f"{key} : Total samples with MSE less than {mse_less_10_thresh}, {value}")
        next_samples = new_samples_ld(df_all_molec, molec_dict, verbose, save_fig, cl_shuffle_seed, gp_shuffle_seed, dist_seed)